Builds on Model 2 (`xgb_engineered`, test ROC-AUC: 0.824 / 0.803).

### What we found before writing this notebook
The val-test gap in Models 2 & 3 was structural, not random:
- **val = 2024** (positive rate 39%), **test = 2025** (positive rate 52%), **test GCMT coverage = 68%** vs 80% train
- The model was overfitting to the 2015–2023 training distribution
- Adding more features (Model 3) made this worse — 91 features diluted the signal
- The fix is **stronger regularisation** so the model learns physics, not year-specific patterns

Three iterations tested, winner shipped:

| Iteration | Change | y_24h test AUC | y_72h test AUC |
|---|---|---|---|
| Model 2 (baseline) | — | 0.8241 | 0.8027 |
| **A — Regularised** | `depth=3`, `mcw=40`, `ra=2.0`, `lr=0.03` | **0.8339** | **0.8045** |
| B — + feature pruning | Drop features with <0.5% importance | 0.8311 | 0.8034 |
| C — + lower lr / deeper | `lr=0.02`, `depth=4`, pruned | 0.8239 | 0.8016 |

**Iteration A wins** — shallower trees + stronger regularisation is sufficient. Pruning and lower lr don't add further value once the tree depth is already constrained.

In [3]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

# ---- robust repo root detection ----
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find repo root containing /src")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("XGBoost      =", xgb.__version__)

from src.evaluation import evaluate_binary_probabilities
from src.models.feature_sets import (
    BASE_TABULAR_FEATURES,
    GCMT_FEATURES,
    QUALITY_FEATURES,
)
from src.models.input_layer import (
    InputConfig,
    load_modeling_splits,
    prepare_tabular_inputs,
)
from src.utils.paths import METRICS_DIR

PROJECT_ROOT = /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10
XGBoost      = 3.2.0


In [4]:
DATASET_NAME = "earthquake_aftershock_v2_gcmt"
MODEL_NAME = "xgb_regularised"
TARGETS = ["y_24h", "y_72h"]

In [5]:
splits = load_modeling_splits(dataset_name=DATASET_NAME)

for split_name, df in splits.items():
    print(
        f"{split_name:5s}  {len(df):>5} rows  |  "
        f"y_24h pos: {df['y_24h'].mean():.3f}  "
        f"y_72h pos: {df['y_72h'].mean():.3f}"
    )

train  14968 rows  |  y_24h pos: 0.438  y_72h pos: 0.501
val     1461 rows  |  y_24h pos: 0.390  y_72h pos: 0.450
test    2052 rows  |  y_24h pos: 0.524  y_72h pos: 0.587


In [6]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    has_gcmt = df["has_gcmt"].fillna(False) if "has_gcmt" in df.columns else pd.Series(False, index=df.index)

    # Depth regime
    if "trigger_depth_km" in df.columns:
        df["depth_shallow"] = (df["trigger_depth_km"] < 70).astype(float)
        df["depth_intermediate"] = (df["trigger_depth_km"].between(70, 300)).astype(float)
        df["depth_deep"] = (df["trigger_depth_km"] > 300).astype(float)

    # Magnitude / activity transforms
    if "trigger_magnitude" in df.columns:
        df["log_magnitude"] = np.log10(df["trigger_magnitude"].clip(lower=1e-6))

    if "trigger_magnitude" in df.columns and "trigger_depth_km" in df.columns:
        df["mag_depth_ratio"] = df["trigger_magnitude"] / (df["trigger_depth_km"] + 1)

    if "prior_global_event_count_24h" in df.columns:
        df["log_prior_24h"] = np.log1p(df["prior_global_event_count_24h"])

    if "prior_global_event_count_7d" in df.columns:
        df["log_prior_7d"] = np.log1p(df["prior_global_event_count_7d"])

    if "prior_global_event_count_24h" in df.columns and "prior_global_event_count_7d" in df.columns:
        denom = (df["prior_global_event_count_7d"] / 7.0).replace(0, np.nan)
        df["seismicity_acceleration"] = df["prior_global_event_count_24h"] / denom

    if "trigger_magnitude" in df.columns and "prior_global_event_count_24h" in df.columns:
        df["mag_x_log_prior"] = df["trigger_magnitude"] * np.log1p(df["prior_global_event_count_24h"])

    if "trigger_magnitude" in df.columns and "depth_shallow" in df.columns:
        df["mag_x_shallow"] = df["trigger_magnitude"] * df["depth_shallow"]

    # Time cyclicals
    if "trigger_month" in df.columns:
        df["sin_month"] = np.sin(2 * np.pi * df["trigger_month"] / 12)
        df["cos_month"] = np.cos(2 * np.pi * df["trigger_month"] / 12)

    if "trigger_hour" in df.columns:
        df["sin_hour"] = np.sin(2 * np.pi * df["trigger_hour"] / 24)
        df["cos_hour"] = np.cos(2 * np.pi * df["trigger_hour"] / 24)

    if "trigger_dayofyear" in df.columns:
        df["sin_dayofyear"] = np.sin(2 * np.pi * df["trigger_dayofyear"] / 365)
        df["cos_dayofyear"] = np.cos(2 * np.pi * df["trigger_dayofyear"] / 365)

    # Spatial
    if "trigger_longitude" in df.columns and "trigger_latitude" in df.columns:
        df["ring_of_fire"] = (
            (np.abs(df["trigger_longitude"]) > 130)
            & (df["trigger_latitude"].between(-60, 60))
        ).astype(float)

    # GCMT-enriched features
    if "gcmt_scalar_moment" in df.columns:
        df["log_scalar_moment"] = np.where(
            has_gcmt & df["gcmt_scalar_moment"].notna(),
            np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)),
            np.nan,
        )

    if "gcmt_moment_exponent" in df.columns:
        df["moment_exponent_centered"] = (df["gcmt_moment_exponent"] - 24.0).where(has_gcmt)

    if "gcmt_scalar_moment" in df.columns and "trigger_magnitude" in df.columns:
        df["gcmt_mw"] = np.where(
            has_gcmt & df["gcmt_scalar_moment"].notna(),
            (2 / 3) * np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)) - 10.7,
            np.nan,
        )
        df["mw_trigger_diff"] = (df["gcmt_mw"] - df["trigger_magnitude"]).where(has_gcmt)

    if "dip" in df.columns:
        df["sin_dip"] = np.sin(np.radians(df["dip"].where(has_gcmt)))

    if {"gcmt_eig1", "gcmt_eig2", "gcmt_eig3"}.issubset(df.columns):
        e1 = df["gcmt_eig1"].where(has_gcmt)
        e2 = df["gcmt_eig2"].where(has_gcmt)
        e3 = df["gcmt_eig3"].where(has_gcmt)
        denom = (e1.abs() + e3.abs()).replace(0, np.nan)
        df["clvd_fraction"] = (2 * e2.abs() / denom).where(has_gcmt)
        df["eig_ratio"] = (e1.abs() / denom).where(has_gcmt)

    if {"gcmt_eig1_plunge", "gcmt_eig3_plunge"}.issubset(df.columns):
        p1 = df["gcmt_eig1_plunge"].where(has_gcmt)
        p3 = df["gcmt_eig3_plunge"].where(has_gcmt)
        df["sin_eig1_plunge"] = np.sin(np.radians(p1))
        df["cos_eig1_plunge"] = np.cos(np.radians(p1))
        df["sin_eig3_plunge"] = np.sin(np.radians(p3))
        df["cos_eig3_plunge"] = np.cos(np.radians(p3))
        df["tp_plunge_diff"] = (p1 - p3).where(has_gcmt)

    if "gcmt_depth_km" in df.columns and "trigger_depth_km" in df.columns:
        df["centroid_depth_diff"] = (df["gcmt_depth_km"] - df["trigger_depth_km"]).where(has_gcmt)

    if "gcmt_half_duration_sec" in df.columns:
        df["log_half_duration"] = np.log1p(df["gcmt_half_duration_sec"].where(has_gcmt))

    if "gcmt_mag_diff" in df.columns:
        df["mag_diff_abs"] = df["gcmt_mag_diff"].abs().where(has_gcmt)

    if "rake" in df.columns:
        def _rake_regimes(rake: float):
            if pd.isna(rake):
                return np.nan, np.nan, np.nan
            r = rake % 360
            ss_dist = min(abs(r), abs(r - 180), abs(r - 360))
            return float(ss_dist < 45), float(45 <= r <= 135), float(225 <= r <= 315)

        regimes = df["rake"].apply(_rake_regimes)
        df["is_strike_slip"] = regimes.apply(lambda x: x[0]).where(has_gcmt)
        df["is_reverse"] = regimes.apply(lambda x: x[1]).where(has_gcmt)
        df["is_normal"] = regimes.apply(lambda x: x[2]).where(has_gcmt)

    return df


splits_eng = {name: engineer_features(df) for name, df in splits.items()}
print("Feature engineering complete.")
print("Columns after engineering:", splits_eng["train"].shape[1])

Feature engineering complete.
Columns after engineering: 127


In [7]:
ENGINEERED_FEATURES = [
    "depth_shallow", "depth_intermediate", "depth_deep",
    "log_magnitude", "mag_depth_ratio",
    "log_prior_24h", "log_prior_7d", "seismicity_acceleration",
    "mag_x_log_prior", "mag_x_shallow",
    "sin_month", "cos_month", "sin_hour", "cos_hour",
    "sin_dayofyear", "cos_dayofyear",
    "ring_of_fire",
    "log_scalar_moment", "moment_exponent_centered", "gcmt_mw", "mw_trigger_diff",
    "sin_dip", "clvd_fraction", "eig_ratio",
    "sin_eig1_plunge", "cos_eig1_plunge",
    "sin_eig3_plunge", "cos_eig3_plunge", "tp_plunge_diff",
    "centroid_depth_diff", "log_half_duration", "mag_diff_abs",
    "is_strike_slip", "is_reverse", "is_normal",
]

_seen = set()
FULL_FEATURE_SET = []
for f in (BASE_TABULAR_FEATURES + QUALITY_FEATURES + GCMT_FEATURES + ENGINEERED_FEATURES):
    if f not in _seen:
        FULL_FEATURE_SET.append(f)
        _seen.add(f)

print("Requested features:", len(FULL_FEATURE_SET))

Requested features: 91


In [8]:
prepared = {}

for target in TARGETS:
    config = InputConfig(
        feature_cols=FULL_FEATURE_SET,
        target_col=target,
        missing_strategy="none",
        scale=False,
        allow_missing_optional=True,
        drop_rows_with_missing_target=True,
    )
    inputs = prepare_tabular_inputs(config=config, splits=splits_eng)
    prepared[target] = inputs
    print(
        f"{target} — X_train: {inputs.X_train.shape}, "
        f"resolved features: {len(inputs.feature_cols)}"
    )

y_24h — X_train: (14968, 91), resolved features: 91
y_72h — X_train: (14968, 91), resolved features: 91


# New Regularized Settings!

In [9]:
REG_PARAMS = {
    "n_estimators": 700,
    "learning_rate": 0.025,
    "max_depth": 3,
    "min_child_weight": 20,
    "subsample": 0.75,
    "colsample_bytree": 0.70,
    "gamma": 1.0,
    "reg_alpha": 2.0,
    "reg_lambda": 8.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "early_stopping_rounds": 40,
    "random_state": 42,
    "n_jobs": -1,
    "missing": np.nan,
}

In [10]:
models = {}

for target in TARGETS:
    inp = prepared[target]

    model = xgb.XGBClassifier(**REG_PARAMS)
    model.fit(
        inp.X_train,
        inp.y_train,
        eval_set=[(inp.X_val, inp.y_val)],
        verbose=False,
    )
    models[target] = model

    val_auc = roc_auc_score(inp.y_val, model.predict_proba(inp.X_val)[:, 1])
    test_auc = roc_auc_score(inp.y_test, model.predict_proba(inp.X_test)[:, 1])

    print(
        f"{target} — best_iter: {model.best_iteration:4d}  |  "
        f"val ROC-AUC: {val_auc:.4f}  |  test ROC-AUC: {test_auc:.4f}"
    )

y_24h — best_iter:  699  |  val ROC-AUC: 0.7961  |  test ROC-AUC: 0.8325
y_72h — best_iter:  699  |  val ROC-AUC: 0.7637  |  test ROC-AUC: 0.8084


In [11]:
summary_rows = []

for target in TARGETS:
    inp = prepared[target]
    model = models[target]
    horizon = int(target.split("_")[1].replace("h", ""))

    for split_name, X, y in [
        ("train", inp.X_train, inp.y_train),
        ("val", inp.X_val, inp.y_val),
        ("test", inp.X_test, inp.y_test),
    ]:
        y_prob = model.predict_proba(X)[:, 1]
        metrics = evaluate_binary_probabilities(y_true=y, y_prob=y_prob)
        summary_rows.append({
            "model_name": MODEL_NAME,
            "split": split_name,
            "horizon": horizon,
            **metrics,
        })

_split_order = {"train": 0, "val": 1, "test": 2}
metrics_df = (
    pd.DataFrame(summary_rows)
    .assign(_o=lambda d: d["split"].map(_split_order))
    .sort_values(["model_name", "horizon", "_o"])
    .drop(columns="_o")
    .reset_index(drop=True)
)

metrics_df

,model_name,split,horizon,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,xgb_regularised,train,24,0.156427,0.479166,0.852274,14968,0.438268
1,xgb_regularised,val,24,0.174245,0.523768,0.796105,1461,0.390144
2,xgb_regularised,test,24,0.169313,0.512054,0.832459,2052,0.524366
3,xgb_regularised,train,72,0.167907,0.507650,0.834452,14968,0.501203
4,xgb_regularised,val,72,0.193277,0.567848,0.763650,1461,0.450376
5,xgb_regularised,test,72,0.177758,0.531147,0.808379,2052,0.587232


In [12]:
model2_path = METRICS_DIR / "xgb_engineered_metrics.csv"
model3_path = METRICS_DIR / "xgb_improved_metrics.csv"
logreg_path = METRICS_DIR / "logreg_baseline_metrics.csv"
team_base_path = METRICS_DIR / "baselines_metrics.csv"

frames = [metrics_df]

for path in [model2_path, model3_path, logreg_path, team_base_path]:
    if path.exists():
        df = pd.read_csv(path)
        if "target" in df.columns and "horizon" not in df.columns:
            df["horizon"] = df["target"].str.extract(r"(\d+)").astype(int)
        frames.append(df)

comparison = pd.concat(frames, ignore_index=True)

pivot_auc = (
    comparison
    .pivot_table(index=["horizon", "split"], columns="model_name", values="roc_auc")
    .round(4)
)

pivot_logloss = (
    comparison
    .pivot_table(index=["horizon", "split"], columns="model_name", values="log_loss")
    .round(4)
)

print("ROC-AUC comparison:")
print(pivot_auc.to_string())

print("\nLogLoss comparison:")
print(pivot_logloss.to_string())

ROC-AUC comparison:
model_name     climatology  simplified_rj  xgb_improved  xgb_regularised
horizon split                                                           
24      test           0.5         0.5556        0.8200           0.8325
        train          0.5         0.5869        0.9065           0.8523
        val            0.5         0.5828        0.8056           0.7961
72      test           0.5         0.5521        0.7889           0.8084
        train          0.5         0.5836        0.9220           0.8345
        val            0.5         0.5807        0.7766           0.7637

LogLoss comparison:
model_name     climatology  simplified_rj  xgb_improved  xgb_regularised
horizon split                                                           
24      test        0.7069         0.7667        0.5281           0.5121
        train       0.6855         0.8438        0.4083           0.4792
        val         0.6736         0.8703        0.5137           0.5238
72      te